# Fase 3 — Siamese dataset construction

In questa fase viene costruito il dataset di coppie necessario per
l'addestramento della rete Siamese.

Le coppie vengono generate separatamente all'interno degli split
`train`, `val` e `test`, mantenendo lo `split_cluster` originale per
evitare leakage tra regioni genomiche correlate.

Una coppia è definita come:

- positiva (`same_disease = 1`) se le due sequenze appartengono alla stessa disease;
- negativa (`same_disease = 0`) se appartengono a disease differenti.

In questa prima versione vengono utilizzate cinque classi:
`healthy`, `gastric cancer`, `ovarian cancer`,
`prostate cancer` e `colorectal cancer`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

METADATA_PATH = (
    PROCESSED_DIR
    / "eccdna_metadata_clean.tsv"
)

metadata = pd.read_csv(
    METADATA_PATH,
    sep="\t",
    low_memory=False
)

print("Shape:", metadata.shape)

Shape: (3755079, 26)


In [3]:
SIAMESE_CLASSES = [
    "healthy",
    "gastric cancer",
    "ovarian cancer",
    "prostate cancer",
    "colorectal cancer"
]

siamese_metadata = metadata[
    metadata["disease_clean"].isin(
        SIAMESE_CLASSES
    )
].copy()

print(
    siamese_metadata["disease_clean"]
    .value_counts()
)

disease_clean
gastric cancer       2593291
healthy               445138
ovarian cancer        172898
prostate cancer       170297
colorectal cancer      88639
Name: count, dtype: int64


Controllo degli split

In [4]:
pd.crosstab(
    siamese_metadata["disease_clean"],
    siamese_metadata["split_cluster"]
)

split_cluster,test,train,val
disease_clean,,,
colorectal cancer,4422,75706,8511
gastric cancer,128668,2212052,252571
healthy,20852,381143,43143
ovarian cancer,8355,147776,16767
prostate cancer,8263,145810,16224


Bilanciamento poichè gastric cancer sono tantissime

In [5]:
N_TRAIN_PER_CLASS = 10_000
N_VAL_PER_CLASS = 2_000
N_TEST_PER_CLASS = 2_000

RANDOM_STATE = 42

In [6]:
def sample_balanced_split(
    df,
    split_name,
    classes,
    n_per_class,
    random_state=42
):
    """
    Costruisce un subset bilanciato per disease
    all'interno di uno specifico split.
    """

    parts = []

    split_df = df[
        df["split_cluster"] == split_name
    ]

    for disease in classes:

        class_df = split_df[
            split_df["disease_clean"] == disease
        ]

        n = min(
            n_per_class,
            len(class_df)
        )

        sampled = class_df.sample(
            n=n,
            random_state=random_state
        )

        parts.append(sampled)

    return pd.concat(
        parts,
        ignore_index=True
    )

In [7]:
train_pool = sample_balanced_split(
    siamese_metadata,
    "train",
    SIAMESE_CLASSES,
    N_TRAIN_PER_CLASS,
    RANDOM_STATE
)

val_pool = sample_balanced_split(
    siamese_metadata,
    "val",
    SIAMESE_CLASSES,
    N_VAL_PER_CLASS,
    RANDOM_STATE
)

test_pool = sample_balanced_split(
    siamese_metadata,
    "test",
    SIAMESE_CLASSES,
    N_TEST_PER_CLASS,
    RANDOM_STATE
)

In [8]:
print("TRAIN")
print(
    train_pool["disease_clean"]
    .value_counts()
)

print("\nVALIDATION")
print(
    val_pool["disease_clean"]
    .value_counts()
)

print("\nTEST")
print(
    test_pool["disease_clean"]
    .value_counts()
)

TRAIN
disease_clean
healthy              10000
gastric cancer       10000
ovarian cancer       10000
prostate cancer      10000
colorectal cancer    10000
Name: count, dtype: int64

VALIDATION
disease_clean
healthy              2000
gastric cancer       2000
ovarian cancer       2000
prostate cancer      2000
colorectal cancer    2000
Name: count, dtype: int64

TEST
disease_clean
healthy              2000
gastric cancer       2000
ovarian cancer       2000
prostate cancer      2000
colorectal cancer    2000
Name: count, dtype: int64


Controllo delle lunghezze

In [9]:
def add_length_bin(df):

    df = df.copy()

    df["length_bin"] = pd.cut(
        df["length"],
        bins=[
            -np.inf,
            300,
            1000,
            np.inf
        ],
        labels=[
            "short",
            "medium",
            "long"
        ],
        right=False
    )

    return df

In [10]:
train_pool = add_length_bin(train_pool)
val_pool = add_length_bin(val_pool)
test_pool = add_length_bin(test_pool)

In [11]:
pd.crosstab(
    train_pool["disease_clean"],
    train_pool["length_bin"],
    normalize="index"
).round(3)

length_bin,short,medium,long
disease_clean,,,
colorectal cancer,0.008,0.713,0.279
gastric cancer,0.014,0.640,0.346
healthy,0.424,0.229,0.347
ovarian cancer,0.656,0.311,0.033
prostate cancer,0.445,0.458,0.096


Costruzione coppie positive. Una coppia positiva è:

A = prostate cancer
B = prostate cancer

same_disease = 1

In [12]:
def generate_positive_pairs(
    pool,
    n_pairs,
    rng
):
    """
    Genera coppie positive campionando prima
    uniformemente la disease.

    In questo modo una classe molto grande non domina
    la generazione delle coppie.
    """

    classes = pool[
        "disease_clean"
    ].unique()

    class_groups = {
        disease: pool[
            pool["disease_clean"] == disease
        ]
        for disease in classes
    }

    pairs = []

    for _ in range(n_pairs):

        disease = rng.choice(classes)

        group = class_groups[disease]

        if len(group) < 2:
            continue

        indices = rng.choice(
            len(group),
            size=2,
            replace=False
        )

        a = group.iloc[indices[0]]
        b = group.iloc[indices[1]]

        pairs.append({
            "id_a": a["id"],
            "id_b": b["id"],

            "disease_a": disease,
            "disease_b": disease,

            "length_a": a["length"],
            "length_b": b["length"],

            "source_a": a["source_db"],
            "source_b": b["source_db"],

            "same_disease": 1
        })

    return pairs

Coppie negative: 

A = healthy
B = ovarian cancer

same_disease = 0

In [13]:
def generate_negative_pairs(
    pool,
    n_pairs,
    rng
):
    """
    Genera coppie negative scegliendo
    uniformemente due disease differenti.
    """

    classes = pool[
        "disease_clean"
    ].unique()

    class_groups = {
        disease: pool[
            pool["disease_clean"] == disease
        ]
        for disease in classes
    }

    pairs = []

    for _ in range(n_pairs):

        disease_a, disease_b = rng.choice(
            classes,
            size=2,
            replace=False
        )

        group_a = class_groups[disease_a]
        group_b = class_groups[disease_b]

        a = group_a.iloc[
            rng.integers(len(group_a))
        ]

        b = group_b.iloc[
            rng.integers(len(group_b))
        ]

        pairs.append({
            "id_a": a["id"],
            "id_b": b["id"],

            "disease_a": disease_a,
            "disease_b": disease_b,

            "length_a": a["length"],
            "length_b": b["length"],

            "source_a": a["source_db"],
            "source_b": b["source_db"],

            "same_disease": 0
        })

    return pairs

Funzione completa per uno split

In [14]:
def build_pairs(
    pool,
    split_name,
    n_positive,
    n_negative,
    seed=42
):
    rng = np.random.default_rng(seed)

    positive = generate_positive_pairs(
        pool,
        n_positive,
        rng
    )

    negative = generate_negative_pairs(
        pool,
        n_negative,
        rng
    )

    pairs = pd.DataFrame(
        positive + negative
    )

    pairs["split"] = split_name

    # Mescola le coppie
    pairs = pairs.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    return pairs

In [15]:
# coppie iniziali

N_TRAIN_POS = 50_000
N_TRAIN_NEG = 50_000

N_VAL_POS = 10_000
N_VAL_NEG = 10_000

N_TEST_POS = 10_000
N_TEST_NEG = 10_000

In [16]:
train_pairs = build_pairs(
    train_pool,
    "train",
    N_TRAIN_POS,
    N_TRAIN_NEG,
    seed=42
)

val_pairs = build_pairs(
    val_pool,
    "val",
    N_VAL_POS,
    N_VAL_NEG,
    seed=43
)

test_pairs = build_pairs(
    test_pool,
    "test",
    N_TEST_POS,
    N_TEST_NEG,
    seed=44
)

In [17]:
train_pairs.head()

,id_a,id_b,disease_a,disease_b,length_a,length_b,source_a,source_b,same_disease,split
0,CircleBaseV2_000749987,CircleBaseV2_002786651,gastric cancer,healthy,582,9490,CircleBaseV2,CircleBaseV2,0,train
1,eccDNABase_000813806,eccDNABase_000254199,healthy,prostate cancer,653,336,eccDNABase,eccDNABase,0,train
2,CircleBaseV2_002977252,CircleBaseV2_002969863,colorectal cancer,colorectal cancer,355,1045,CircleBaseV2,CircleBaseV2,1,train
3,eccDNABase_000861092,CircleBaseV2_002855583,prostate cancer,healthy,8513,424,eccDNABase,CircleBaseV2,0,train
4,CircleBaseV2_002026958,eccDNABase_000294419,gastric cancer,prostate cancer,762,193,CircleBaseV2,eccDNABase,0,train


In [18]:
print(train_pairs.shape)
print(val_pairs.shape)
print(test_pairs.shape)

(100000, 10)
(20000, 10)
(20000, 10)


In [19]:
print(
    train_pairs["same_disease"]
    .value_counts()
)

print(
    val_pairs["same_disease"]
    .value_counts()
)

print(
    test_pairs["same_disease"]
    .value_counts()
)

same_disease
0    50000
1    50000
Name: count, dtype: int64
same_disease
1    10000
0    10000
Name: count, dtype: int64
same_disease
0    10000
1    10000
Name: count, dtype: int64


Verifica delle coppie

In [20]:
def validate_pairs(pairs):

    positive_errors = pairs[
        (pairs["same_disease"] == 1)
        &
        (pairs["disease_a"] != pairs["disease_b"])
    ]

    negative_errors = pairs[
        (pairs["same_disease"] == 0)
        &
        (pairs["disease_a"] == pairs["disease_b"])
    ]

    identical_ids = pairs[
        pairs["id_a"] == pairs["id_b"]
    ]

    print(
        "Positive pair errors:",
        len(positive_errors)
    )

    print(
        "Negative pair errors:",
        len(negative_errors)
    )

    print(
        "Pairs with identical IDs:",
        len(identical_ids)
    )

In [21]:
print("TRAIN")
validate_pairs(train_pairs)

print("\nVAL")
validate_pairs(val_pairs)

print("\nTEST")
validate_pairs(test_pairs)

TRAIN
Positive pair errors: 0
Negative pair errors: 0
Pairs with identical IDs: 0

VAL
Positive pair errors: 0
Negative pair errors: 0
Pairs with identical IDs: 0

TEST
Positive pair errors: 0
Negative pair errors: 0
Pairs with identical IDs: 0


Controllo leakage tra split

In [22]:
train_ids = set(
    train_pairs["id_a"]
).union(
    train_pairs["id_b"]
)

val_ids = set(
    val_pairs["id_a"]
).union(
    val_pairs["id_b"]
)

test_ids = set(
    test_pairs["id_a"]
).union(
    test_pairs["id_b"]
)

print(
    "Train ∩ Val:",
    len(train_ids & val_ids)
)

print(
    "Train ∩ Test:",
    len(train_ids & test_ids)
)

print(
    "Val ∩ Test:",
    len(val_ids & test_ids)
)

Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


Analisi positive pairs per disease

In [23]:
positive_distribution = (
    train_pairs[
        train_pairs["same_disease"] == 1
    ]
    ["disease_a"]
    .value_counts()
)

positive_distribution

disease_a
ovarian cancer       10171
prostate cancer      10010
healthy               9982
gastric cancer        9958
colorectal cancer     9879
Name: count, dtype: int64

Combinazioni in negative pairs

In [24]:
negative_pairs = train_pairs[
    train_pairs["same_disease"] == 0
].copy()

negative_pairs["class_pair"] = negative_pairs.apply(
    lambda row: " | ".join(
        sorted([
            row["disease_a"],
            row["disease_b"]
        ])
    ),
    axis=1
)

negative_pairs[
    "class_pair"
].value_counts()

class_pair
healthy | prostate cancer              5093
colorectal cancer | healthy            5077
colorectal cancer | ovarian cancer     5056
ovarian cancer | prostate cancer       5047
healthy | ovarian cancer               5005
colorectal cancer | prostate cancer    4977
colorectal cancer | gastric cancer     4967
gastric cancer | healthy               4957
gastric cancer | prostate cancer       4951
gastric cancer | ovarian cancer        4870
Name: count, dtype: int64

Problema del source db

In [25]:
train_pairs["same_source"] = (
    train_pairs["source_a"]
    == train_pairs["source_b"]
).astype(int)

pd.crosstab(
    train_pairs["same_disease"],
    train_pairs["same_source"],
    normalize="index"
).round(3)

same_source,0,1
same_disease,,
0,0.588,0.412
1,0.141,0.859


Differenza di lunghezza

In [26]:
train_pairs["length_diff"] = (
    train_pairs["length_a"]
    - train_pairs["length_b"]
).abs()

In [27]:
train_pairs.groupby(
    "same_disease"
)["length_diff"].describe()

,count,mean,std,min,25%,50%,75%,max
same_disease,,,,,,,,
0,50000.0,2721.46902,12997.117615,0.0,185.0,462.0,1007.0,196584.0
1,50000.0,2323.08236,11830.237863,0.0,109.0,296.0,804.0,197061.0


# Salavatggio

In [28]:
SIAMESE_DIR = (
    PROCESSED_DIR
    / "siamese"
)

SIAMESE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [29]:
train_pairs.to_csv(
    SIAMESE_DIR / "train_pairs.tsv",
    sep="\t",
    index=False
)

val_pairs.to_csv(
    SIAMESE_DIR / "val_pairs.tsv",
    sep="\t",
    index=False
)

test_pairs.to_csv(
    SIAMESE_DIR / "test_pairs.tsv",
    sep="\t",
    index=False
)

print("Siamese pair datasets saved.")

Siamese pair datasets saved.


In [30]:
def add_pair_key(df):
    df = df.copy()

    df["pair_key"] = df.apply(
        lambda row: "||".join(
            sorted([
                str(row["id_a"]),
                str(row["id_b"])
            ])
        ),
        axis=1
    )

    return df

In [31]:
train_pairs = add_pair_key(train_pairs)

print(
    "Duplicate train pairs:",
    train_pairs["pair_key"].duplicated().sum()
)

Duplicate train pairs: 8


# Fase 3B — Controlled Siamese pairs

La prima generazione di coppie ha evidenziato un possibile confondente
legato a `source_db`: le coppie positive appartenevano molto più spesso
alla stessa sorgente rispetto alle coppie negative.

Per ridurre questa scorciatoia, viene costruita una seconda versione
del dataset Siamese imponendo che entrambe le sequenze della coppia
appartengano alla stessa `source_db`.

Viene inoltre richiesto che le sequenze appartengano alla stessa
fascia di lunghezza (`length_bin`), per ridurre le differenze di
sparsità FCGR direttamente riconducibili alla lunghezza.

Le coppie vengono sempre generate separatamente all'interno degli
split train, validation e test.

In [32]:
def add_length_bin(df):
    df = df.copy()

    df["length_bin"] = pd.cut(
        df["length"],
        bins=[
            -np.inf,
            300,
            1000,
            np.inf
        ],
        labels=[
            "short",
            "medium",
            "long"
        ],
        right=False
    )

    return df


train_pool = add_length_bin(train_pool)
val_pool = add_length_bin(val_pool)
test_pool = add_length_bin(test_pool)

In [33]:
print(
    train_pool[
        ["disease_clean", "source_db", "length_bin"]
    ].value_counts().head(30)
)

disease_clean      source_db     length_bin
colorectal cancer  CircleBaseV2  medium        7005
ovarian cancer     eccDNABase    short         6554
gastric cancer     CircleBaseV2  medium        6358
prostate cancer    eccDNABase    medium        4398
                                 short         4094
gastric cancer     CircleBaseV2  long          3458
ovarian cancer     eccDNABase    medium        3088
colorectal cancer  CircleBaseV2  long          2737
healthy            CircleBaseV2  short         2182
                   eccDNABase    short         2058
                                 long          1778
                   CircleBaseV2  long          1692
                                 medium        1175
                   eccDNABase    medium        1115
prostate cancer    eccDNABase    long           733
                   CircleBaseV2  short          358
ovarian cancer     eccDNABase    long           323
prostate cancer    CircleBaseV2  long           230
                    

Funzione per creare chiave della coppia per evitare duplicati come A-B e B-A

In [34]:
def make_pair_key(id_a, id_b):
    return "||".join(
        sorted([
            str(id_a),
            str(id_b)
        ])
    )

Positive pairs controllate

In [35]:
def generate_controlled_positive_pairs(
    pool,
    n_pairs,
    rng
):
    """
    Coppie positive controllate:

    - stessa disease
    - stessa source_db
    - stesso length_bin
    - ID differenti
    - nessuna coppia duplicata
    """

    group_cols = [
        "disease_clean",
        "source_db",
        "length_bin"
    ]

    groups = {
        key: group.reset_index(drop=True)
        for key, group in pool.groupby(
            group_cols,
            observed=True
        )
        if len(group) >= 2
    }

    valid_keys = list(groups.keys())

    pairs = []
    used_pairs = set()

    attempts = 0
    max_attempts = n_pairs * 50

    while (
        len(pairs) < n_pairs
        and attempts < max_attempts
    ):

        attempts += 1

        key_index = rng.integers(
            len(valid_keys)
        )

        disease, source, length_bin = (
            valid_keys[key_index]
        )

        group = groups[
            (disease, source, length_bin)
        ]

        idx = rng.choice(
            len(group),
            size=2,
            replace=False
        )

        a = group.iloc[idx[0]]
        b = group.iloc[idx[1]]

        pair_key = make_pair_key(
            a["id"],
            b["id"]
        )

        if pair_key in used_pairs:
            continue

        used_pairs.add(pair_key)

        pairs.append({
            "id_a": a["id"],
            "id_b": b["id"],

            "disease_a": disease,
            "disease_b": disease,

            "source_a": source,
            "source_b": source,

            "length_a": a["length"],
            "length_b": b["length"],

            "length_bin_a": length_bin,
            "length_bin_b": length_bin,

            "same_disease": 1,

            "pair_key": pair_key
        })

    if len(pairs) < n_pairs:
        print(
            f"WARNING: requested {n_pairs:,} positive pairs, "
            f"generated {len(pairs):,}"
        )

    return pairs

Negative pairs controllate

In [36]:
def generate_controlled_negative_pairs(
    pool,
    n_pairs,
    rng
):
    """
    Coppie negative controllate:

    - disease differenti
    - stessa source_db
    - stesso length_bin
    - nessuna coppia duplicata
    """

    grouped = {}

    for (
        source,
        length_bin,
        disease
    ), group in pool.groupby(
        [
            "source_db",
            "length_bin",
            "disease_clean"
        ],
        observed=True
    ):

        grouped.setdefault(
            (source, length_bin),
            {}
        )

        grouped[
            (source, length_bin)
        ][disease] = (
            group.reset_index(drop=True)
        )

    valid_strata = [
        key
        for key, diseases in grouped.items()
        if len(diseases) >= 2
    ]

    pairs = []
    used_pairs = set()

    attempts = 0
    max_attempts = n_pairs * 50

    while (
        len(pairs) < n_pairs
        and attempts < max_attempts
    ):

        attempts += 1

        stratum = valid_strata[
            rng.integers(
                len(valid_strata)
            )
        ]

        source, length_bin = stratum

        disease_groups = grouped[stratum]

        diseases = list(
            disease_groups.keys()
        )

        disease_a, disease_b = rng.choice(
            diseases,
            size=2,
            replace=False
        )

        group_a = disease_groups[
            disease_a
        ]

        group_b = disease_groups[
            disease_b
        ]

        a = group_a.iloc[
            rng.integers(len(group_a))
        ]

        b = group_b.iloc[
            rng.integers(len(group_b))
        ]

        pair_key = make_pair_key(
            a["id"],
            b["id"]
        )

        if pair_key in used_pairs:
            continue

        used_pairs.add(pair_key)

        pairs.append({
            "id_a": a["id"],
            "id_b": b["id"],

            "disease_a": disease_a,
            "disease_b": disease_b,

            "source_a": source,
            "source_b": source,

            "length_a": a["length"],
            "length_b": b["length"],

            "length_bin_a": length_bin,
            "length_bin_b": length_bin,

            "same_disease": 0,

            "pair_key": pair_key
        })

    if len(pairs) < n_pairs:
        print(
            f"WARNING: requested {n_pairs:,} negative pairs, "
            f"generated {len(pairs):,}"
        )

    return pairs

Funzione completa controlled

In [37]:
def build_controlled_pairs(
    pool,
    split_name,
    n_positive,
    n_negative,
    seed=42
):

    rng = np.random.default_rng(seed)

    positive = (
        generate_controlled_positive_pairs(
            pool,
            n_positive,
            rng
        )
    )

    negative = (
        generate_controlled_negative_pairs(
            pool,
            n_negative,
            rng
        )
    )

    pairs = pd.DataFrame(
        positive + negative
    )

    pairs["split"] = split_name

    pairs = pairs.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    return pairs

Generazione nuovi dataste

In [38]:
controlled_train_pairs = (
    build_controlled_pairs(
        train_pool,
        "train",
        n_positive=50_000,
        n_negative=50_000,
        seed=42
    )
)

controlled_val_pairs = (
    build_controlled_pairs(
        val_pool,
        "val",
        n_positive=10_000,
        n_negative=10_000,
        seed=43
    )
)

controlled_test_pairs = (
    build_controlled_pairs(
        test_pool,
        "test",
        n_positive=10_000,
        n_negative=10_000,
        seed=44
    )
)

In [39]:
print(
    "TRAIN:",
    controlled_train_pairs.shape
)

print(
    "VAL:",
    controlled_val_pairs.shape
)

print(
    "TEST:",
    controlled_test_pairs.shape
)

TRAIN: (100000, 13)
VAL: (20000, 13)
TEST: (20000, 13)


Controllo positive/negative

In [40]:
for name, pairs in [
    ("TRAIN", controlled_train_pairs),
    ("VAL", controlled_val_pairs),
    ("TEST", controlled_test_pairs)
]:

    print("\n", name)

    print(
        pairs["same_disease"]
        .value_counts()
    )


 TRAIN
same_disease
0    50000
1    50000
Name: count, dtype: int64

 VAL
same_disease
1    10000
0    10000
Name: count, dtype: int64

 TEST
same_disease
0    10000
1    10000
Name: count, dtype: int64


Verifica delle condizioni controlled

In [41]:
def validate_controlled_pairs(pairs):

    positive_errors = pairs[
        (pairs["same_disease"] == 1)
        &
        (pairs["disease_a"] != pairs["disease_b"])
    ]

    negative_errors = pairs[
        (pairs["same_disease"] == 0)
        &
        (pairs["disease_a"] == pairs["disease_b"])
    ]

    identical_ids = pairs[
        pairs["id_a"] == pairs["id_b"]
    ]

    different_source = pairs[
        pairs["source_a"]
        != pairs["source_b"]
    ]

    different_length_bin = pairs[
        pairs["length_bin_a"]
        != pairs["length_bin_b"]
    ]

    duplicate_pairs = (
        pairs["pair_key"]
        .duplicated()
        .sum()
    )

    print(
        "Positive label errors:",
        len(positive_errors)
    )

    print(
        "Negative label errors:",
        len(negative_errors)
    )

    print(
        "Identical IDs:",
        len(identical_ids)
    )

    print(
        "Different source:",
        len(different_source)
    )

    print(
        "Different length bin:",
        len(different_length_bin)
    )

    print(
        "Duplicate pairs:",
        duplicate_pairs
    )

In [42]:
print("TRAIN")
validate_controlled_pairs(
    controlled_train_pairs
)

print("\nVAL")
validate_controlled_pairs(
    controlled_val_pairs
)

print("\nTEST")
validate_controlled_pairs(
    controlled_test_pairs
)

TRAIN
Positive label errors: 0
Negative label errors: 0
Identical IDs: 0
Different source: 0
Different length bin: 0
Duplicate pairs: 0

VAL
Positive label errors: 0
Negative label errors: 0
Identical IDs: 0
Different source: 0
Different length bin: 0
Duplicate pairs: 0

TEST
Positive label errors: 0
Negative label errors: 0
Identical IDs: 0
Different source: 0
Different length bin: 0
Duplicate pairs: 0


Controllo leakage

In [43]:
controlled_train_ids = set(
    controlled_train_pairs["id_a"]
).union(
    controlled_train_pairs["id_b"]
)

controlled_val_ids = set(
    controlled_val_pairs["id_a"]
).union(
    controlled_val_pairs["id_b"]
)

controlled_test_ids = set(
    controlled_test_pairs["id_a"]
).union(
    controlled_test_pairs["id_b"]
)

print(
    "Train ∩ Val:",
    len(
        controlled_train_ids
        & controlled_val_ids
    )
)

print(
    "Train ∩ Test:",
    len(
        controlled_train_ids
        & controlled_test_ids
    )
)

print(
    "Val ∩ Test:",
    len(
        controlled_val_ids
        & controlled_test_ids
    )
)

Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


Sorurce db diventa inutile

In [44]:
controlled_train_pairs[
    "same_source"
] = (
    controlled_train_pairs["source_a"]
    ==
    controlled_train_pairs["source_b"]
).astype(int)

In [45]:
pd.crosstab(
    controlled_train_pairs[
        "same_disease"
    ],
    controlled_train_pairs[
        "same_source"
    ],
    normalize="index"
)

same_source,1
same_disease,
0,1.0
1,1.0


Controllo lunghezza

In [46]:
controlled_train_pairs[
    "length_diff"
] = (
    controlled_train_pairs[
        "length_a"
    ]
    -
    controlled_train_pairs[
        "length_b"
    ]
).abs()

In [47]:
controlled_train_pairs.groupby(
    "same_disease"
)["length_diff"].describe()

,count,mean,std,min,25%,50%,75%,max
same_disease,,,,,,,,
0,50000.0,3726.32716,15858.771651,0.0,55.0,147.0,494.0,196565.0
1,50000.0,3239.02026,16260.675621,0.0,52.0,147.0,448.0,196568.0


Confronto naive vs controlled

In [48]:
naive_length = (
    train_pairs
    .groupby("same_disease")[
        "length_diff"
    ]
    .median()
    .rename("naive")
)

controlled_length = (
    controlled_train_pairs
    .groupby("same_disease")[
        "length_diff"
    ]
    .median()
    .rename("controlled")
)

length_comparison = pd.concat(
    [
        naive_length,
        controlled_length
    ],
    axis=1
)

length_comparison

,naive,controlled
same_disease,,
0,462.0,147.0
1,296.0,147.0


In [49]:
naive_source = pd.crosstab(
    train_pairs["same_disease"],
    train_pairs["same_source"],
    normalize="index"
)

controlled_source = pd.crosstab(
    controlled_train_pairs[
        "same_disease"
    ],
    controlled_train_pairs[
        "same_source"
    ],
    normalize="index"
)

print("NAIVE")
display(naive_source)

print("CONTROLLED")
display(controlled_source)

NAIVE


same_source,0,1
same_disease,,
0,0.58774,0.41226
1,0.14080,0.85920


CONTROLLED


same_source,1
same_disease,
0,1.0
1,1.0


Controllo distribuzione disease psoitive

In [50]:
controlled_positive_distribution = (
    controlled_train_pairs[
        controlled_train_pairs[
            "same_disease"
        ] == 1
    ]
    ["disease_a"]
    .value_counts()
)

controlled_positive_distribution

disease_a
healthy              13306
prostate cancer      13112
colorectal cancer     8879
gastric cancer        7829
ovarian cancer        6874
Name: count, dtype: int64

Controllo combinazioni negative

In [51]:
controlled_negative = (
    controlled_train_pairs[
        controlled_train_pairs[
            "same_disease"
        ] == 0
    ]
    .copy()
)

controlled_negative[
    "class_pair"
] = controlled_negative.apply(
    lambda row: " | ".join(
        sorted([
            row["disease_a"],
            row["disease_b"]
        ])
    ),
    axis=1
)

controlled_negative[
    "class_pair"
].value_counts()

class_pair
colorectal cancer | healthy            5437
healthy | ovarian cancer               5375
healthy | prostate cancer              5357
colorectal cancer | prostate cancer    5344
gastric cancer | healthy               5270
gastric cancer | prostate cancer       5110
ovarian cancer | prostate cancer       4773
colorectal cancer | ovarian cancer     4615
gastric cancer | ovarian cancer        4448
colorectal cancer | gastric cancer     4271
Name: count, dtype: int64

In [52]:
print(
    "Numero combinazioni negative:",
    controlled_negative[
        "class_pair"
    ].nunique()
)

controlled_negative[
    "class_pair"
].value_counts(
    normalize=True
).mul(100).round(2)

Numero combinazioni negative: 10


class_pair
colorectal cancer | healthy            10.87
healthy | ovarian cancer               10.75
healthy | prostate cancer              10.71
colorectal cancer | prostate cancer    10.69
gastric cancer | healthy               10.54
gastric cancer | prostate cancer       10.22
ovarian cancer | prostate cancer        9.55
colorectal cancer | ovarian cancer      9.23
gastric cancer | ovarian cancer         8.90
colorectal cancer | gastric cancer      8.54
Name: proportion, dtype: float64

Salvataggio controlled

In [53]:
CONTROLLED_DIR = (
    PROCESSED_DIR
    / "siamese"
    / "controlled"
)

CONTROLLED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [54]:
controlled_train_pairs.to_csv(
    CONTROLLED_DIR
    / "train_pairs.tsv",
    sep="\t",
    index=False
)

controlled_val_pairs.to_csv(
    CONTROLLED_DIR
    / "val_pairs.tsv",
    sep="\t",
    index=False
)

controlled_test_pairs.to_csv(
    CONTROLLED_DIR
    / "test_pairs.tsv",
    sep="\t",
    index=False
)

print(
    "Controlled Siamese datasets saved."
)

Controlled Siamese datasets saved.


In [55]:
validate_controlled_pairs(controlled_train_pairs)

pd.crosstab(
    controlled_train_pairs["same_disease"],
    controlled_train_pairs["same_source"],
    normalize="index"
)

controlled_train_pairs.groupby(
    "same_disease"
)["length_diff"].describe()

controlled_positive_distribution

controlled_negative[
    "class_pair"
].value_counts()

Positive label errors: 0
Negative label errors: 0
Identical IDs: 0
Different source: 0
Different length bin: 0
Duplicate pairs: 0


class_pair
colorectal cancer | healthy            5437
healthy | ovarian cancer               5375
healthy | prostate cancer              5357
colorectal cancer | prostate cancer    5344
gastric cancer | healthy               5270
gastric cancer | prostate cancer       5110
ovarian cancer | prostate cancer       4773
colorectal cancer | ovarian cancer     4615
gastric cancer | ovarian cancer        4448
colorectal cancer | gastric cancer     4271
Name: count, dtype: int64

# Fase 3C
Nuova funzione positive bilanciata

In [56]:
def generate_balanced_controlled_positive_pairs(
    pool,
    n_pairs_per_class,
    classes,
    rng
):
    """
    Genera lo stesso numero di coppie positive
    per ogni disease.

    Vincoli:
    - stessa disease
    - stessa source_db
    - stesso length_bin
    - ID differenti
    - nessun duplicato
    """

    all_pairs = []
    used_pairs = set()

    for disease in classes:

        disease_pool = pool[
            pool["disease_clean"] == disease
        ]

        groups = {
            key: group.reset_index(drop=True)
            for key, group in disease_pool.groupby(
                ["source_db", "length_bin"],
                observed=True
            )
            if len(group) >= 2
        }

        valid_keys = list(groups.keys())

        if not valid_keys:
            print(
                f"WARNING: no valid groups for {disease}"
            )
            continue

        disease_pairs = []

        attempts = 0
        max_attempts = (
            n_pairs_per_class * 100
        )

        while (
            len(disease_pairs)
            < n_pairs_per_class
            and attempts < max_attempts
        ):

            attempts += 1

            key = valid_keys[
                rng.integers(
                    len(valid_keys)
                )
            ]

            source, length_bin = key

            group = groups[key]

            idx = rng.choice(
                len(group),
                size=2,
                replace=False
            )

            a = group.iloc[idx[0]]
            b = group.iloc[idx[1]]

            pair_key = make_pair_key(
                a["id"],
                b["id"]
            )

            if pair_key in used_pairs:
                continue

            used_pairs.add(pair_key)

            disease_pairs.append({
                "id_a": a["id"],
                "id_b": b["id"],

                "disease_a": disease,
                "disease_b": disease,

                "source_a": source,
                "source_b": source,

                "length_a": a["length"],
                "length_b": b["length"],

                "length_bin_a": length_bin,
                "length_bin_b": length_bin,

                "same_disease": 1,

                "pair_key": pair_key
            })

        if len(disease_pairs) < n_pairs_per_class:

            print(
                f"WARNING: {disease} requested "
                f"{n_pairs_per_class:,}, generated "
                f"{len(disease_pairs):,}"
            )

        all_pairs.extend(
            disease_pairs
        )

    return all_pairs

In [57]:
def build_final_controlled_pairs(
    pool,
    split_name,
    classes,
    n_positive_per_class,
    n_negative,
    seed=42
):

    rng = np.random.default_rng(seed)

    positive = (
        generate_balanced_controlled_positive_pairs(
            pool,
            n_positive_per_class,
            classes,
            rng
        )
    )

    negative = (
        generate_controlled_negative_pairs(
            pool,
            n_negative,
            rng
        )
    )

    pairs = pd.DataFrame(
        positive + negative
    )

    pairs["split"] = split_name

    pairs = pairs.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    return pairs

In [58]:
# Train

final_train_pairs = build_final_controlled_pairs(
    train_pool,
    "train",
    SIAMESE_CLASSES,
    n_positive_per_class=10_000,
    n_negative=50_000,
    seed=42
)

In [59]:
# validation

final_val_pairs = build_final_controlled_pairs(
    val_pool,
    "val",
    SIAMESE_CLASSES,
    n_positive_per_class=2_000,
    n_negative=10_000,
    seed=43
)

In [60]:
# test

final_test_pairs = build_final_controlled_pairs(
    test_pool,
    "test",
    SIAMESE_CLASSES,
    n_positive_per_class=2_000,
    n_negative=10_000,
    seed=44
)

In [61]:
final_positive_distribution = (
    final_train_pairs[
        final_train_pairs["same_disease"] == 1
    ]
    ["disease_a"]
    .value_counts()
)

final_positive_distribution

disease_a
gastric cancer       10000
ovarian cancer       10000
colorectal cancer    10000
healthy              10000
prostate cancer      10000
Name: count, dtype: int64

In [62]:
validate_controlled_pairs(
    final_train_pairs
)

Positive label errors: 0
Negative label errors: 0
Identical IDs: 0
Different source: 0
Different length bin: 0
Duplicate pairs: 0


In [63]:
final_train_pairs["same_source"] = (
    final_train_pairs["source_a"]
    ==
    final_train_pairs["source_b"]
).astype(int)

pd.crosstab(
    final_train_pairs["same_disease"],
    final_train_pairs["same_source"],
    normalize="index"
)

same_source,1
same_disease,
0,1.0
1,1.0


In [69]:
# Mappa ID -> tissue dai metadata originali

id_to_tissue = (
    metadata
    .set_index(
        metadata["id"].astype(str)
    )["tissue"]
    .to_dict()
)

In [70]:
def add_tissue_information(pairs):

    pairs = pairs.copy()

    pairs["tissue_a"] = (
        pairs["id_a"]
        .astype(str)
        .map(id_to_tissue)
    )

    pairs["tissue_b"] = (
        pairs["id_b"]
        .astype(str)
        .map(id_to_tissue)
    )

    pairs["same_tissue"] = (
        pairs["tissue_a"]
        ==
        pairs["tissue_b"]
    ).astype(int)

    return pairs

In [71]:
final_train_pairs = add_tissue_information(
    final_train_pairs
)

final_val_pairs = add_tissue_information(
    final_val_pairs
)

final_test_pairs = add_tissue_information(
    final_test_pairs
)

In [72]:
for name, pairs in [
    ("TRAIN", final_train_pairs),
    ("VAL", final_val_pairs),
    ("TEST", final_test_pairs)
]:

    print(f"\n{name}")

    print(
        "Missing tissue A:",
        pairs["tissue_a"].isna().sum()
    )

    print(
        "Missing tissue B:",
        pairs["tissue_b"].isna().sum()
    )


TRAIN
Missing tissue A: 3680
Missing tissue B: 3654

VAL
Missing tissue A: 784
Missing tissue B: 799

TEST
Missing tissue A: 781
Missing tissue B: 808


In [73]:
def classify_tissue_match(row):

    if (
        pd.isna(row["tissue_a"])
        or
        pd.isna(row["tissue_b"])
    ):
        return "missing"

    if row["tissue_a"] == row["tissue_b"]:
        return "same"

    return "different"

In [74]:
for pairs in [
    final_train_pairs,
    final_val_pairs,
    final_test_pairs
]:

    pairs["tissue_match"] = (
        pairs.apply(
            classify_tissue_match,
            axis=1
        )
    )

In [75]:
tissue_crosstab = pd.crosstab(
    final_train_pairs[
        "same_disease"
    ],
    final_train_pairs[
        "tissue_match"
    ],
    normalize="index"
)

tissue_crosstab.round(3)

tissue_match,different,missing,same
same_disease,,,
0,0.925,0.074,0.001
1,0.257,0.062,0.682


In [76]:
for name, pairs in [
    ("TRAIN", final_train_pairs),
    ("VAL", final_val_pairs),
    ("TEST", final_test_pairs)
]:

    print(f"\n{name}")

    display(
        pd.crosstab(
            pairs["same_disease"],
            pairs["tissue_match"],
            normalize="index"
        ).round(3)
    )


TRAIN


tissue_match,different,missing,same
same_disease,,,
0,0.925,0.074,0.001
1,0.257,0.062,0.682



VAL


tissue_match,different,missing,same
same_disease,,,
0,0.917,0.082,0.001
1,0.262,0.064,0.674



TEST


tissue_match,different,missing,same
same_disease,,,
0,0.918,0.081,0.001
1,0.266,0.066,0.668


In [78]:
print("Tissue più frequenti - sequence A")

display(
    final_train_pairs[
        "tissue_a"
    ]
    .value_counts(
        dropna=False
    )
    .head(20)
)

Tissue più frequenti - sequence A


tissue_a
Stomach                                                           19465
Muscle                                                            11769
Colorectal tumor tissue                                           11466
ES2                                                                8828
Prostate                                                           8300
Colon                                                              8000
OVCAR8                                                             5697
Ovary                                                              4894
LnCap                                                              4626
NaN                                                                3680
C4-2                                                               3347
Subcutaneous Adipose                                               2570
Urine                                                              1834
Plasma                                                 

In [77]:
print("Tissue più frequenti - sequence B")

display(
    final_train_pairs[
        "tissue_b"
    ]
    .value_counts(
        dropna=False
    )
    .head(20)
)

Tissue più frequenti - sequence B


tissue_b
Stomach                                                           19626
Colorectal tumor tissue                                           11644
Muscle                                                            11433
ES2                                                                8726
Prostate                                                           8436
Colon                                                              8115
OVCAR8                                                             5808
Ovary                                                              4839
LnCap                                                              4651
NaN                                                                3654
C4-2                                                               3293
Subcutaneous Adipose                                               2502
Urine                                                              1781
Plasma                                                 

In [79]:
disease_tissue = pd.crosstab(
    siamese_metadata[
        "disease_clean"
    ],
    siamese_metadata[
        "tissue"
    ],
    normalize="index"
)

disease_tissue

tissue,"22RV1, LNCaP",22Rv1,Brain,C15,C4-2,C4-2B,C4-2R,"C4-2R, C4-2",COLO320,COLO320DM,...,Sperm,Stomach,Subcutaneous Adipose,Testicle,Urine,WTC11,cardiac,human adipose stem cells,ovary,plasma
disease_clean,,,,,,,,,,,,,,,,,,,,,
colorectal cancer,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000011,0.000293,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
gastric cancer,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.999997,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
healthy,0.000000,0.000000,0.000694,0.000442,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.040749,0.000000,0.109759,0.003547,0.066508,0.000175,0.004946,0.004537,0.000000,0.000026
ovarian cancer,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000046,0.000000
prostate cancer,0.000006,0.000006,0.000000,0.000000,0.251879,0.006641,0.006383,0.000006,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.021981,0.000000,0.000000,0.000000,0.000000,0.000000


In [80]:
for disease in SIAMESE_CLASSES:

    print(
        f"\n{disease.upper()}"
    )

    display(
        siamese_metadata[
            siamese_metadata[
                "disease_clean"
            ] == disease
        ]
        ["tissue"]
        .value_counts(
            normalize=True,
            dropna=False
        )
        .head(10)
        .mul(100)
        .round(2)
    )


HEALTHY


tissue
Muscle                                                            59.08
Subcutaneous Adipose                                              10.31
Plasma                                                             7.77
Urine                                                              6.25
NaN                                                                6.06
Sperm                                                              3.83
Plasma leukocytes                                                  1.74
Control(HCT116, 293A and 293T cells absorbing HaloTag plasmid)     0.83
HEK293                                                             0.79
Health                                                             0.46
Name: proportion, dtype: float64


GASTRIC CANCER


tissue
Stomach    100.0
SNU-16       0.0
Name: proportion, dtype: float64


OVARIAN CANCER


tissue
ES2            66.13
OVCAR8         33.15
Ovary           0.32
FT190           0.24
FT194           0.15
ovary           0.00
OVCAR-8         0.00
NCI-ADR-RES     0.00
Caov-3          0.00
SKOV3           0.00
Name: proportion, dtype: float64


PROSTATE CANCER


tissue
LnCap       49.59
C4-2        24.70
Prostate    10.97
PC-3         8.55
Urine        2.16
NaN          1.93
Lncap        0.78
C4-2B        0.65
C4-2R        0.63
PC3          0.03
Name: proportion, dtype: float64


COLORECTAL CANCER


tissue
Colorectal tumor tissue    98.05
Colon                       1.91
COLO320DM                   0.03
HCT116                      0.01
COLO320                     0.00
Name: proportion, dtype: float64

In [81]:
pd.crosstab(
    [
        final_train_pairs[
            "same_disease"
        ],
        final_train_pairs[
            "source_a"
        ]
    ],
    final_train_pairs[
        "tissue_match"
    ],
    normalize="index"
).round(3)

tissue_match               different  missing   same
same_disease source_a                               
0            CircleBaseV2      0.852    0.148  0.000
             eccDNABase        0.999    0.000  0.001
1            CircleBaseV2      0.070    0.123  0.807
             eccDNABase        0.445    0.000  0.555

In [82]:
print("Positive pairs per disease - TRAIN")

display(
    final_train_pairs[
        final_train_pairs[
            "same_disease"
        ] == 1
    ]
    ["disease_a"]
    .value_counts()
)

Positive pairs per disease - TRAIN


disease_a
gastric cancer       10000
ovarian cancer       10000
colorectal cancer    10000
healthy              10000
prostate cancer      10000
Name: count, dtype: int64

In [83]:
print("Positive pairs per disease - VAL")

display(
    final_val_pairs[
        final_val_pairs[
            "same_disease"
        ] == 1
    ]
    ["disease_a"]
    .value_counts()
)

Positive pairs per disease - VAL


disease_a
ovarian cancer       2000
prostate cancer      2000
gastric cancer       2000
healthy              2000
colorectal cancer    2000
Name: count, dtype: int64

In [84]:
FINAL_DIR = (
    PROCESSED_DIR
    / "siamese"
    / "final"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Final Siamese directory:")
print(FINAL_DIR)

Final Siamese directory:
..\data\processed\siamese\final


In [85]:
final_train_pairs.to_csv(
    FINAL_DIR / "train_pairs.tsv",
    sep="\t",
    index=False
)

final_val_pairs.to_csv(
    FINAL_DIR / "val_pairs.tsv",
    sep="\t",
    index=False
)

final_test_pairs.to_csv(
    FINAL_DIR / "test_pairs.tsv",
    sep="\t",
    index=False
)

print("Final Siamese datasets saved.")

Final Siamese datasets saved.


In [66]:
for filename in [
    "train_pairs.tsv",
    "val_pairs.tsv",
    "test_pairs.tsv"
]:
    path = FINAL_DIR / filename

    print(
        filename,
        "->",
        path.exists(),
        "|",
        round(path.stat().st_size / (1024**2), 2),
        "MB"
    )

train_pairs.tsv -> True | 16.54 MB
val_pairs.tsv -> True | 3.24 MB
test_pairs.tsv -> True | 3.26 MB


In [86]:
final_dataset_summary = pd.DataFrame({
    "split": [
        "train",
        "val",
        "test"
    ],
    "n_pairs": [
        len(final_train_pairs),
        len(final_val_pairs),
        len(final_test_pairs)
    ],
    "positive_pairs": [
        (final_train_pairs["same_disease"] == 1).sum(),
        (final_val_pairs["same_disease"] == 1).sum(),
        (final_test_pairs["same_disease"] == 1).sum()
    ],
    "negative_pairs": [
        (final_train_pairs["same_disease"] == 0).sum(),
        (final_val_pairs["same_disease"] == 0).sum(),
        (final_test_pairs["same_disease"] == 0).sum()
    ]
})

final_dataset_summary

,split,n_pairs,positive_pairs,negative_pairs
0,train,100000,50000,50000
1,val,20000,10000,10000
2,test,20000,10000,10000


In [87]:
final_dataset_summary.to_csv(
    FINAL_DIR / "dataset_summary.csv",
    index=False
)

# Conclusioni della fase 3 — Costruzione del dataset Siamese

In questa fase è stato costruito e validato il dataset di coppie destinato all'addestramento della rete Siamese.

Le coppie sono state generate separatamente all'interno degli split `train`, `validation` e `test`, mantenendo lo `split_cluster` originale. Questo garantisce che gli stessi identificativi non vengano condivisi tra gli insiemi e riduce il rischio di information leakage tra sequenze genomicamente correlate.

La prima versione del dataset, basata su un campionamento casuale delle coppie, ha evidenziato due potenziali scorciatoie: la sorgente dei dati (`source_db`) e la differenza di lunghezza tra le sequenze. In particolare, le coppie positive appartenevano molto più frequentemente alla stessa sorgente rispetto alle coppie negative.

Per questo motivo è stata sviluppata una seconda versione controlled del dataset, nella quale entrambe le sequenze di ogni coppia devono appartenere alla stessa `source_db` e alla stessa fascia di lunghezza (`short`, `medium` o `long`). Le coppie positive appartengono alla stessa disease, mentre quelle negative appartengono a disease differenti.

La versione finale contiene:

* 100.000 coppie nel training set;
* 20.000 coppie nel validation set;
* 20.000 coppie nel test set.

Ogni split è perfettamente bilanciato tra coppie positive e negative. Inoltre, nel training set le 50.000 coppie positive sono equamente distribuite tra le cinque classi considerate, con 10.000 coppie positive per disease. La stessa strategia di bilanciamento è stata applicata a validation e test.

I controlli finali hanno verificato:

* assenza di errori nelle label delle coppie;
* assenza di coppie con identificativi identici;
* assenza di coppie duplicate;
* assenza di sovrapposizioni tra gli ID di train, validation e test;
* stessa `source_db` per entrambe le sequenze di ogni coppia;
* stessa fascia di lunghezza per entrambe le sequenze di ogni coppia;
* bilanciamento 50/50 tra coppie positive e negative.

Il controllo sulla differenza di lunghezza ha mostrato che il matching per `length_bin` riduce significativamente la differenza tra coppie positive e negative, limitando la possibilità che la rete utilizzi la sola sparsità della FCGR come scorciatoia.

È stato inoltre analizzato il possibile confondente associato al `tissue`. Tale analisi ha mostrato che le coppie positive appartengono molto più frequentemente allo stesso tissue rispetto alle coppie negative. Questo comportamento è coerente con la struttura biologica e con la composizione del dataset, poiché alcune disease risultano fortemente associate a specifici tissue o cell line.

Il tissue viene quindi considerato un confondente residuo noto. Non viene imposto un ulteriore matching sul tissue nella baseline, poiché questo potrebbe ridurre eccessivamente le combinazioni negative disponibili e produrre un dataset artificiale. Il suo effetto verrà invece considerato nella fase di valutazione attraverso stress test o protocolli tissue-aware.

La versione finale del dataset Siamese è stata salvata in:

`data/processed/siamese/final/`

con i file:

* `train_pairs.tsv`
* `val_pairs.tsv`
* `test_pairs.tsv`
* `dataset_summary.csv`

La fase successiva sarà dedicata alla costruzione e all'addestramento della rete Siamese sulle rappresentazioni FCGR. Verranno inizialmente confrontati i valori di `k = 5`, `k = 6` e `k = 7`, selezionati nella fase precedente come i compromessi più promettenti tra densità della rappresentazione, specificità dei k-mer e costo computazionale.
